In [ ]:
import pandas as pd

In [20]:
airbnb_listing = pd.read_csv(r'C:\Users\prith\Downloads\airbnb-data-pipeline\airbnb-data-pipeline\data\processed\featured_engineered_data.csv')

In [21]:
airbnb_listing.shape

(20521, 43)

In [22]:
airbnb_listing.dtypes

hosts_time_as_host_years          float64
host_is_superhost                    bool
neighbourhood_cleansed                str
neighbourhood_group_cleansed          str
latitude                          float64
longitude                         float64
property_type                         str
room_type                             str
accommodates                        int64
bathrooms                         float64
bedrooms                          float64
price                             float64
maximum_nights                    float64
maximum_nights_avg_ntm            float64
has_availability                     bool
number_of_reviews                 float64
estimated_occupancy_l365d           int64
days_since_first_review           float64
days_since_last_review            float64
has_reviews                         int64
review_scores_cleanliness         float64
review_scores_location            float64
calculated_host_listings_count      int64
amenity_count                     

In [23]:
#its a medium dataset so we will keep: 80% train, 10% validation and 10% test

In [24]:
from sklearn.model_selection import train_test_split

#features and target
X = airbnb_listing.drop(columns=['price', 'log_price'])
y = airbnb_listing['log_price'] #log price is used as its evenly spread

#train (80%) and temp (20%)
# X <-> all the columns the model learns from, the inputs, the clues.
# Y <-> the column the model is trying to predict, the answer
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=airbnb_listing['neighbourhood_group_cleansed'] #as its a strong point and categorial
)

#temp into val (10%) and test (10%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=X_temp['neighbourhood_group_cleansed']
)

print("Train:", X_train.shape)
print("Val:  ", X_val.shape)
print("Test: ", X_test.shape)

Train: (16416, 41)
Val:   (2052, 41)
Test:  (2053, 41)


In [25]:
# # A little theory i had to refer to. (used claude for the theory and for understanding the topic)

# Train set — this is what the model learns from. It sees both X and y during training. It adjusts itself to predict y from X as accurately as possible.

# Validation set — the model never trains on this. You use it to tune settings (called hyperparameters). Like asking a student practice questions they haven't seen before to check if they're actually learning or just memorizing.

# Test set — the final exam. Completely untouched until the very end. You use it exactly once to measure real performance. If you used it multiple times you'd accidentally start optimizing for it, which is cheating.

# Why did we drop price AND log_price from X?

# Because X should never contain the answer. If you left price in X, the model would just memorize "price = price" and learn nothing useful. That's called target leakage — the answer leaking into the features.

# Why stratify by neighbourhood_group_cleansed?

# Without stratification, random splitting might give you:
# Train: 80% Manhattan listings, 20% Brooklyn
# Test:  20% Manhattan listings, 80% Brooklyn

In [26]:
#   full dataset (20,521 rows)
#          ↓
#     ┌────┴────┐
#   X (features)   y (target = log_price)
#   41 columns      1 column
#          ↓
#   Split into 3:
#   ┌──────────────────────────────────┐
#   │ X_train / y_train  (16,416 rows) │ ← model learns here
#   ├──────────────────────────────────┤
#   │ X_val / y_val      (2,052 rows)  │ ← tune settings here
#   ├──────────────────────────────────┤
#   │ X_test / y_test    (2,053 rows)  │ ← final exam here
#   └──────────────────────────────────┘

In [27]:
X_train.to_csv(r'../data/output/X_train.csv', index=False)
X_val.to_csv(r'../data/output/X_val.csv', index=False)
X_test.to_csv(r'../data/output/X_test.csv', index=False)

y_train.to_csv(r'../data/output/y_train.csv', index=False)
y_val.to_csv(r'../data/output/y_val.csv', index=False)
y_test.to_csv(r'../data/output/y_test.csv', index=False)